# Importing a public dataset : feeding state and larval locomotion

This notebook is a complete entry point to **larvaworld**. Starting from nothing but a public
download link, it walks through the whole path an experimentalist takes with real tracking data :

1. **get** a published dataset of *Drosophila* larva locomotion,
2. **inspect** its raw, tracker-specific file format,
3. **convert** that format into the one larvaworld reads,
4. **import** it, which also derives all the secondary locomotory metrics,
5. **analyse and visualize** it, comparing three experimental groups,
6. **replay** the recorded tracks as videos.

You do not need to know anything about the internals of larvaworld to follow it. Every
configuration object is introduced at the moment it is needed and only for what it does here.

Steps 1-4 are a **one-off cost**. Once a dataset has been imported it is stored in the larvaworld
format and reloaded in a second, in this or any future session.

> This is the first of a series of tutorials importing datasets from public repositories. The
> recipe below - *look at the raw layout, map it onto a lab format, convert, import* - is the same
> one you will apply to your own tracker output.

## The biological question

The goal is to compare the locomotion patterns of larvae subjected to different diets over
5 hours :

| group | diet | metabolic state |
|---|---|---|
| **Fed** | normal food | fed |
| **Sucrose** | sucrose only | protein-deprived |
| **Starved** | nothing | starved |

The distinct metabolic states of the three larva groups might have an impact on their locomotion.
We specifically focus on the **temporal evolution of their dispersal in space** : how far the
larvae of each group get away from where they started, and how that distance grows over time.

We will build up to that answer in increasingly informative steps :

1. plots of the raw trajectories,
2. boxplots of endpoint dispersal metrics (mean, final, maximum) - the endpoint situation only,
3. timeplots of dispersal - the full timecourse, with mean and variance,
4. replay videos of the trajectories aligned at a common origin, combined side by side.

## Setup

Importing larvaworld initializes its configuration registry : some components are loaded from
disc and the rest are built on the fly. `VERBOSE = 1` makes the import report its progress, which
is useful for a step that takes a few minutes.

In [ ]:
%matplotlib inline

from pathlib import Path

from IPython.display import display

import larvaworld
from larvaworld.lib import reg, sim, util
from larvaworld.lib.process import convert_spine_files_to_per_parameter_txt
from larvaworld.lib.reg.generators import ReplayConf

larvaworld.VERBOSE = 1

# Where the figures and videos produced by this notebook are written.
MEDIA_DIR = Path("./media/feeding_state")
plot_dir = (MEDIA_DIR / "plots").as_posix()
video_dir = (MEDIA_DIR / "videos").as_posix()

# Rendering the replay videos needs ffmpeg and takes several minutes. Turn it on when you
# have run everything else successfully.
MAKE_VIDEOS = False

# Set to True to rewrite the converted raw files even if they already exist.
FORCE_CONVERSION = False

ds = []  # the imported datasets, filled in further below

## Step 1 - Get the data

The dataset is openly available on Zenodo, published alongside Jovanic et al. (2025). Both are
cited in full at the end of this notebook.

- **Record** : <https://zenodo.org/records/15075754>
- **File to download** : [`Main_Locomotion.rar`](https://zenodo.org/records/15075754/files/Main_Locomotion.rar?download=1)

**What to do :**

1. Download `Main_Locomotion.rar` from the link above.
2. Extract it. It is a `.rar` archive, so you need 7-Zip, WinRAR or `unrar` - larvaworld does not
   unpack archives for you.
3. Point `DOWNLOAD_ROOT` below at the extracted `Main_Locomotion` folder.

The extracted archive looks like this. This notebook uses only the three feeding-state groups
under `Figure1/1a-d` :

```text
Main_Locomotion/
└── Locomotion/
    ├── Figure1/1a-d/
    │   ├── Fed/       <one folder per recording session>
    │   ├── Sucrose/   <one folder per recording session>
    │   └── Starved/   <one folder per recording session>
    └── Figure6/6g-h/  (a different experiment, not used here)
```

Note that the archive is large (several GB) because it ships the full tracker output, including
video frames. We will only read one file type out of it.

In [ ]:
# ---- EDIT THIS LINE if you extracted the archive somewhere else ----
DOWNLOAD_ROOT = Path.home() / "Downloads" / "Main_Locomotion"
# -------------------------------------------------------------------

SOURCE_ROOT = DOWNLOAD_ROOT / "Locomotion" / "Figure1" / "1a-d"

# The three experimental groups, and the color each one gets in every plot of this notebook.
palette = {"Fed": "black", "Sucrose": "red", "Starved": "purple"}
gIDs = list(palette)

# The raw spine files of each group, one per recording session.
spine_files = {
    gID: sorted(p.as_posix() for p in SOURCE_ROOT.glob(f"{gID}/*/*.spine"))
    for gID in gIDs
}

DATA_AVAILABLE = all(len(v) > 0 for v in spine_files.values())

if not DATA_AVAILABLE:
    print(f"Raw data not found under :\n  {SOURCE_ROOT}\n")
    print(
        "Download and extract Main_Locomotion.rar as described above, then set DOWNLOAD_ROOT."
    )
    print("The rest of the notebook will be skipped until then.")
else:
    for gID, files in spine_files.items():
        size = sum(Path(f).stat().st_size for f in files) / 1e6
        print(f"{gID:9s} : {len(files)} recordings, {size:6.1f} MB of spine data")
        for f in files:
            print(f"            {Path(f).parent.name}")

## Step 2 - Look at the raw data

Each recording session folder holds the complete output of the tracker used in the source lab :

| file | content | needed here |
|---|---|---|
| `.spine` | the **midline coordinates** of every tracked larva, frame by frame | **yes** |
| `.outline` | the full body contour, frame by frame | no |
| `.blobs` | raw blob detection data | no |
| `.summary` | per-frame tracker statistics | no |
| `.dat` | a single derived per-frame quantity | no |
| `.set` | the tracker's acquisition settings (XML) | no |
| `.png` | reference frames of the recorded dish | no |

Everything this notebook needs is in the `.spine` file. Let's look at the first lines of one.

In [ ]:
if DATA_AVAILABLE:
    example = Path(spine_files["Fed"][0])
    print(f"{example.parent.name} / {example.name}\n")
    with example.open() as f:
        for _ in range(3):
            print(f.readline().rstrip())

A `.spine` file has no header and no fixed delimiter width. Its columns are :

| column | content |
|---|---|
| 0 | the recording tag, identical on every row of the file |
| 1 | the **track id**, an integer that is unique *within this recording only* |
| 2 | the **time** `t` in seconds since the start of the recording |
| 3 ... 24 | the **11 midline points**, as *interleaved* `x1 y1 x2 y2 ... x11 y11` pairs, in **mm** |

Point 1 is the head and point 11 is the tail.

Two facts about this layout matter for what comes next :

- the coordinates are **interleaved** in x/y pairs, and
- the track ids **repeat across recordings** - the larva numbered `1` in one session is a
  different animal from the larva numbered `1` in the next.

Let's confirm the scale of the data and check that the numbers are physically sensible.

In [ ]:
if DATA_AVAILABLE:
    import numpy as np
    import pandas as pd

    df = pd.read_csv(example, sep=r"\s+", header=None)
    xs, ys = df.iloc[:, 3::2].values, df.iloc[:, 4::2].values
    seg = np.sqrt(np.diff(xs, axis=1) ** 2 + np.diff(ys, axis=1) ** 2)
    dts = df[2].diff()

    print(f"rows (larva x frame) : {df.shape[0]}")
    print(f"tracked larvae       : {df[1].nunique()}")
    print(f"recording duration   : {df[2].max():.0f} s")
    print(f"mean sampling step   : {dts[dts > 0].mean():.3f} s")
    print(
        f"coordinate range     : {min(xs.min(), ys.min()):.1f} - {max(xs.max(), ys.max()):.1f} mm"
    )
    print(f"mean body length     : {seg.sum(axis=1).mean():.2f} mm")

A mean body length of about 4 mm is exactly right for a third-instar larva, which confirms the
coordinates are in **millimetres**. The coordinate range fills a square arena of roughly
193 mm per side. Both of these will have to agree with the configuration we use to import.

Note also that the sampling step is not perfectly constant - this tracker records at a variable
frame rate, which the import handles.

## Step 3 - What larvaworld expects

Raw data comes in as many formats as there are labs. larvaworld handles this with the
**`LabFormat`** configuration : one stored entry per lab, describing how that lab's files are laid
out, what the tracker records, and how the data must be preprocessed.

The dataset we are importing was recorded in the Jovanic lab, so we use the stored `Jovanic`
format. It already knows the arena, the number of midline points and the units.

In [ ]:
Jovanic_lf = reg.conf.LabFormat.get("Jovanic")

# The tracker timestep. This dataset was recorded at a variable rate averaging ~0.09 s, and we
# resample it onto a regular 0.1 s grid during the import.
Jovanic_lf.tracker.dt = 0.1

print("Available lab formats :", reg.conf.LabFormat.confIDs, "\n")
print(f"midline points  : {Jovanic_lf.tracker.Npoints}")
print(
    f"contour points  : {Jovanic_lf.tracker.Ncontour}   (this tracker's contours are not imported)"
)
print(f"coordinate unit : {Jovanic_lf.tracker.XY_unit}")
print(f"timestep        : {Jovanic_lf.tracker.dt} s")
print(
    f"arena           : {Jovanic_lf.env_params.arena.dims} m, {Jovanic_lf.env_params.arena.geometry}"
)
print(
    f"preprocessing   : rescale by {Jovanic_lf.preprocess.rescale_by} (mm -> m), "
    f"low-pass filter at {Jovanic_lf.preprocess.filter_f} Hz, "
    f"transposition '{Jovanic_lf.preprocess.transposition}'"
)

Everything matches what we measured in the raw file : 11 midline points, millimetres, and a
0.193 x 0.193 m arena for coordinates spanning 0-192 mm. Good - the stored format describes this
dataset correctly, and we do not have to define one ourselves.

Now for the file layout. Each lab format declares a `filesystem` structure. The Jovanic format is
`per_parameter` : instead of one file per larva, it expects **one file per recorded quantity**,
with all larvae of a group stacked in it.

In [ ]:
# Where the raw and imported data of this lab format live. Both are exposed here in case you
# prefer to keep them outside the larvaworld package directory.
RAW_FOLDER = Jovanic_lf.raw_folder
PROC_FOLDER = Jovanic_lf.processed_folder

# The name we give this experiment. It becomes the folder holding the converted raw files.
EXP = "FeedingState"

print(f"file structure : '{Jovanic_lf.filesystem.structure}'\n")
print(f"raw data is read from       : {RAW_FOLDER}/{EXP}/")
print(f"imported data is written to : {PROC_FOLDER}/{EXP}/\n")
print("The four files expected per group, e.g. for 'Fed' :")
for suf in ["larvaid", "t", "x_spine", "y_spine"]:
    print(f"   Fed_{suf}.txt")
print("\n   ..._larvaid.txt   1 column  : which larva each row belongs to")
print("   ..._t.txt         1 column  : the timestamp of each row, in seconds")
print("   ..._x_spine.txt  11 columns : the x coordinate of each midline point")
print("   ..._y_spine.txt  11 columns : the y coordinate of each midline point")
print("\nAll of them tab-separated and without a header.")

### The gap to close

Comparing what we have with what is expected gives us the exact list of things the conversion has
to do :

| the raw tracker output | what larvaworld reads | so we must |
|---|---|---|
| one `.spine` file per recording session | one set of files per **group** | concatenate the sessions of each group |
| track ids unique only within a session | ids unique across everything we compare | offset each session's ids, and each group's, before concatenating |
| `x` and `y` interleaved in pairs | all `x` in one file, all `y` in another | de-interleave the coordinate columns |
| a leading recording-tag column | no such column | drop it |
| whitespace-separated | tab-separated | rewrite with the right delimiter |
| bare integer track ids | string agent ids, as everywhere else in larvaworld | prefix them with `Larva_` |

That is the whole of it. Note what is *not* on the list : nothing about units, filtering,
resampling, arena geometry or derived metrics. All of that is handled by the lab format during the
import, and it is why the conversion below is so short.

A fifth file, `..._state.txt`, holding behavioral state annotations, is optional in this format.
The raw tracker does not produce one, so we do not write it and the import simply proceeds
without it.

## Step 4 - Convert the raw files (runs once)

`convert_spine_files_to_per_parameter_txt` does exactly the five things listed above. It is part
of larvaworld, so you can call it on any tracker output that shares this `.spine` layout.

The conversion is skipped if the output files already exist, so re-running this cell is cheap.
Set `FORCE_CONVERSION = True` at the top of the notebook to rewrite them.

In [ ]:
if DATA_AVAILABLE:
    target_dir = f"{RAW_FOLDER}/{EXP}"
    for i, gID in enumerate(gIDs):
        res = convert_spine_files_to_per_parameter_txt(
            source_files=spine_files[gID],
            target_dir=target_dir,
            source_id=gID,
            Npoints=Jovanic_lf.tracker.Npoints,
            # Keep the larva IDs distinct between the three groups as well, so that the
            # comparative plots further below can pool them without collisions.
            id_base=(i + 1) * 1000000,
            overwrite=FORCE_CONVERSION,
        )
        print(
            f"{gID:9s} : {res['files']} recordings -> {res['rows']:7d} rows, {res['tracks']:4d} tracks"
        )

    print(f"\nWritten to {target_dir}")
    for f in sorted(Path(target_dir).glob("*.txt")):
        print(f"   {f.name:22s} {f.stat().st_size / 1e6:7.1f} MB")

## Step 5 - Import

The import converts the raw data into the larvaworld format and, in the same pass, derives all the
secondary metrics we will need for the analysis. It is configured by three groups of arguments.

**Which tracks to keep.** Not every detected track is usable : the tracker loses and re-acquires
animals, and very short fragments carry no information about dispersal.

In [ ]:
constraints = util.AttrDict(
    {
        # Use the tracker's own identities rather than re-linking broken tracks.
        "match_ids": False,
        # Resample the variable-rate recording onto the regular dt grid set above.
        "interpolate_ticks": True,
        # Discard tracks shorter than this.
        "min_duration_in_sec": 20,
        # Keep only the first minute of each recording, the window we analyse.
        "time_slice": (0, 60),
    }
)

**What to compute.** Enrichment turns raw coordinates into interpretable quantities : body bending
and orientation (`angular`), velocities and displacement (`spatial`), and a segmentation of the
track into behavioral bouts such as crawling strides and pauses (`bout_detection`).

The `dsp_*` settings are the ones that matter for our question : they ask for **dispersal from the
starting point**, measured from second 0, over windows of 40 and 60 seconds.

In [ ]:
enr_kws = util.AttrDict(
    {
        "proc_keys": ["angular", "spatial"],
        "anot_keys": ["bout_detection"],
        # Also store a copy of every trajectory translated to start at the origin.
        "traj2origin": True,
        "tor_durs": [20],
        "dsp_starts": [0],
        "dsp_stops": [40, 60],
    }
)

**What to import.** One dataset per group, each with its own ID, color and reference ID. The
reference ID is the handle you use to reload the dataset later, from anywhere.

In [ ]:
refIDs = [f"{EXP}.{gID}" for gID in gIDs]

kws = {
    "parent_dir": EXP,
    "source_ids": gIDs,
    "refIDs": refIDs,
    "colors": [palette[gID] for gID in gIDs],
    "merged": False,
    "save_dataset": True,
    "enrich_conf": enr_kws,
    **constraints,
}

The next cell does the actual work. **It takes a few minutes** - it reads over a million rows,
resamples them, and computes the full metric set for each of the three groups.

You only ever need to run it once. Afterwards the datasets live in the `processed` folder and are
reloaded in a second by the cell after it.

For reference, with the constraints above this dataset yields roughly **143 / 144 / 138** larvae
for Fed / Sucrose / Starved.

In [ ]:
if DATA_AVAILABLE:
    ds = Jovanic_lf.import_datasets(**kws)

    for d in ds:
        print(f"{d.id:9s} : {d.config.N} larvae, stored at {d.config.dir}")

## Step 6 - Reloading in a later session

From now on you never touch the raw data again. In any future session, skip everything above and
start here : `reg.loadRef` fetches an imported dataset by its reference ID.

In [ ]:
if not ds:
    if all(refID in reg.conf.Ref.confIDs for refID in refIDs):
        ds = [reg.loadRef(id=refID, load=True) for refID in refIDs]
        print("Loaded :", [d.id for d in ds])
    else:
        print("These datasets have not been imported yet. Run steps 1-5 first.")

## Step 7 - Analysis

larvaworld ships a library of plotting routines, each registered under a short name. You pick one
by name and hand it the datasets you want compared - the group colors and labels are taken from
the datasets themselves, so every figure is consistent.

In [ ]:
# The available plots, by their unique IDs
print(reg.graphs.ks)

In [ ]:
# Arguments shared by every plot below. Figures are also written to `plot_dir`.
plot_kws = {"datasets": ds, "save_to": plot_dir, "show": False, "subfolder": None}

### The trajectories

First, simply what the larvae did : their paths inside the dish over the analysed minute.

In [ ]:
if ds:
    display(reg.graphs.run("trajectories", **plot_kws))

The same trajectories, but each one translated so that it starts at the origin, and colored by
group. This removes the arbitrary starting position of each animal and makes the *shape and
extent* of the paths directly comparable between groups.

In [ ]:
if ds:
    display(
        reg.graphs.run("trajectories", mode="origin", single_color=True, **plot_kws)
    )

### Endpoint metrics

A boxplot of endpoint metrics - one value per larva, summarising its whole track. Each plot
routine has a sensible default selection, but you can always name the metrics you want by their
short keys :

| key | metric |
|---|---|
| `l` | body length |
| `fsv` | crawling frequency |
| `sv_mu` | mean scaled crawling speed |
| `str_sd_mu` | mean scaled distance covered per stride |
| `run_tr`, `pau_tr` | fraction of time spent running / pausing |
| `tor20_mu` | mean tortuosity over 20 s windows |
| `dsp_0_40_fin` | dispersal reached after 40 s |
| `b_mu`, `bv_mu` | mean body bending and bending velocity |

In [ ]:
if ds:
    display(
        reg.graphs.run(
            "endpoint box",
            ks=[
                "l",
                "fsv",
                "sv_mu",
                "str_sd_mu",
                "run_tr",
                "pau_tr",
                "tor20_mu",
                "dsp_0_40_fin",
                "b_mu",
                "bv_mu",
            ],
            **plot_kws,
        )
    )

And a composite figure summarising exploration behavior across the three groups.

In [ ]:
if ds:
    display(reg.graphs.run("exploration summary", **plot_kws))

### Dispersal : the question we came for

Dispersal is the distance of a larva from where it started. We now compare the three groups on it,
in three increasingly informative ways.

**1. As an endpoint statistic.** The mean, final and maximum dispersal reached during the first
60 seconds - one number per larva, summarised as a boxplot per group. This captures the outcome
but says nothing about how it was reached.

In [ ]:
if ds:
    display(
        reg.graphs.run(
            "endpoint box",
            ks=["dsp_0_60_mu", "dsp_0_60_fin", "dsp_0_60_max"],
            **plot_kws,
        )
    )

**2. As a timecourse.** The dispersal of the larvae from their starting point plotted against
time, showing both the mean and the variance of each group. This is the temporal evolution we are
after : it shows not just how far the groups got, but how fast, and how consistently.

In [ ]:
if ds:
    # The default time range is 0-40 seconds.
    display(reg.graphs.run("dispersal", **plot_kws))

In [ ]:
if ds:
    # The same over the full analysed minute.
    display(reg.graphs.run("dispersal", range=(0, 60), **plot_kws))

The summary versions place the timecourse next to the corresponding trajectories, which makes the
link between the curve and the actual paths immediate.

In [ ]:
if ds:
    display(reg.graphs.run("dispersal summary", **plot_kws))

In [ ]:
if ds:
    display(reg.graphs.run("dispersal summary", range=(0, 60), **plot_kws))

## Step 8 - Replay the recorded tracks as video

A *replay* is a simulation whose agents are driven by recorded data instead of a model. It gives
you the same visualization tools you would use on a simulation - here, the trajectories of all
larvae of a group, transposed to a common origin and drawn as accumulating trails.

Rendering needs `ffmpeg` (installed with larvaworld via `imageio_ffmpeg`) and takes a few minutes
per group, so it is off by default. Set `MAKE_VIDEOS = True` at the top of the notebook to run it.

In [ ]:
def run_replay(d):
    """Render one dataset's tracks to a video file in `video_dir`."""
    screen_kws = {
        "vis_mode": "video",
        "show_display": False,
        "draw_contour": False,
        "draw_midline": False,
        "draw_centroid": False,
        "visible_trails": True,
        "save_video": True,
        "fps": 1,
        "video_file": d.id,
        "media_dir": video_dir,
    }
    replay_conf = ReplayConf(
        transposition="origin", time_range=(0, 60), track_point=d.c.point_idx
    ).nestedConf
    rep = sim.ReplayRun(
        dataset=d,
        parameters=replay_conf,
        id=f"{d.id}_replay",
        screen_kws=screen_kws,
    )
    return rep.run()

In [ ]:
if MAKE_VIDEOS and ds:
    for d in ds:
        run_replay(d)

Finally the three videos are stacked side by side into a single one, giving a direct visual
comparison of how the three metabolic states explore space.

In [ ]:
if MAKE_VIDEOS and ds:
    from larvaworld.lib.util.combining import combine_videos

    combine_videos(file_dir=video_dir, save_as="3conditions.mp4")
    print(f"Written to {video_dir}/3conditions.mp4")

## Where to go next

**Import your own data.** The recipe in this notebook generalises. Start by finding the shipped
`LabFormat` closest to your tracker (`reg.conf.LabFormat.confIDs`) :

| format | raw layout |
|---|---|
| `Jovanic` | one file per recorded quantity, all larvae stacked - what we used here |
| `Schleyer` | one file per larva, plus per-dish metadata |
| `Berni`, `Arguello` | one file per larva, columns in a fixed declared sequence |

If one of them matches your tracker, convert your files to its layout as we did above. If none
does, create a new `LabFormat` describing your tracker and register it - see
`import_datasets.ipynb` for the class and the registry in detail.

**Continue with these datasets.** They are now ordinary larvaworld reference datasets, so you can
use them to calibrate and evaluate models, or as the reference for simulated larva groups.
See `model_evaluation.ipynb` and `replay.ipynb`.

**Work in the browser instead.** Everything above is also available without writing code. Launch
the portal with `larvaworld-portal` and use the dataset and analysis apps.

## References

The dataset used in this notebook and the study it belongs to :

> Jovanic, T. *et al.* Feeding-state dependent neuropeptidergic modulation of reciprocally
> interconnected inhibitory neurons biases sensorimotor decisions in *Drosophila*.
> *Nature Communications* (2025). <https://doi.org/10.1038/s41467-025-61805-y>

> Jovanic, T., & Manceau, D. (2025). *Feeding-state dependent neuropeptidergic modulation of
> reciprocally interconnected inhibitory neurons biases sensorimotor decisions in Drosophila*
> [Dataset]. Zenodo. <https://doi.org/10.5281/zenodo.15075754>

Please cite both if you use this data.